# GeoIntel RAG — QLoRA Fine-tuning on Llama 3.1 8B

Fine-tunes `meta-llama/Llama-3.1-8B-Instruct` on the 2023 Turkey-Syria
earthquake humanitarian response corpus using QLoRA.

**What this notebook does:**
1. Installs dependencies
2. Loads `chunks.jsonl` (your ReliefWeb text chunks)
3. Formats them into (question, context, answer) training examples
4. Loads Llama 3.1 8B in 4-bit quantisation to fit on a free T4 GPU
5. Attaches LoRA adapters — the only weights that will be trained
6. Trains with SFTTrainer (supervised fine-tuning)
7. Pushes the adapter to your HuggingFace Hub

**Note:** This notebook is a standalone fine-tuning experiment.
The production app uses the base Llama 3.1 8B model via the Groq API.
Groq does not support custom adapters; to serve a fine-tuned model
you would deploy via HuggingFace Inference Endpoints or a similar service.

**Runtime:** GPU (T4). Go to Runtime → Change runtime type → T4 GPU before running.

**Expected training time:** ~45-90 min on a free T4 depending on dataset size.

## Cell 1 — Install dependencies

- `transformers` — loads Llama 3.1 and handles tokenisation
- `peft` — adds LoRA adapters to the frozen model
- `bitsandbytes` — enables 4-bit quantisation (makes the 8B model fit in 16 GB VRAM)
- `trl` — provides SFTTrainer, which handles the supervised fine-tuning loop
- `accelerate` — distributed training utilities used by the trainer internally
- `datasets` — HuggingFace Dataset class for efficient batching

In [ ]:
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    trl==0.10.1 \
    accelerate==0.34.2 \
    datasets==2.21.0

## Cell 2 — Authenticate with HuggingFace

Your HF token is needed to:
- Download the gated Llama 3.1 weights (requires accepting Meta's licence on HF Hub)
- Push the trained adapter back to your Hub at the end

Store your token in Colab Secrets (the key icon in the left sidebar) as `HF_TOKEN`.
Never paste tokens directly into notebook cells — they get saved in the file.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

HF_USERNAME = "your-hf-username"  # <-- change this
ADAPTER_REPO = f"{HF_USERNAME}/geointel-llama3-adapter"

## Cell 3 — Upload and load chunks.jsonl

`chunks.jsonl` is the file produced by the ingestion pipeline.
Each line is one text chunk from a ReliefWeb situation report.

Upload it using the file picker below, then we load and inspect it.

In [ ]:
from google.colab import files

print("Upload chunks.jsonl from data/processed/ in your local project")
uploaded = files.upload()  # opens a file picker in Colab

In [ ]:
import json

chunks = []
with open("chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded {len(chunks)} chunks")
print(f"\nExample chunk:")
print(f"  source:  {chunks[0]['source']}")
print(f"  title:   {chunks[0]['metadata'].get('title', 'N/A')}")
print(f"  text:    {chunks[0]['text'][:200]}...")

## Cell 4 — Format chunks into training examples

Each training example needs three things:
- **Question** — derived from the report title and any location mentioned
- **Context** — the chunk text (what the model is allowed to read)
- **Answer** — key factual sentences extracted from the chunk

We wrap all three in Llama 3.1's chat template:
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system}<|eot_id|><|start_header_id|>user<|end_header_id|>

{context + question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{answer}<|eot_id|>
```

The model is trained to predict the assistant's tokens given the user's tokens.
It never has to predict the question — only the answer.

In [ ]:
import re

PROVINCES = [
    "Hatay", "Kahramanmaras", "Gaziantep", "Adiyaman", "Malatya",
    "Osmaniye", "Adana", "Sanliurfa", "Diyarbakir", "Kilis", "Elazig",
    "Idlib", "Aleppo", "Syria", "Turkey", "Turkiye",
]

SYSTEM_PROMPT = (
    "You are a humanitarian intelligence assistant specialising in "
    "disaster response for the 2023 Turkey-Syria earthquake. "
    "Answer using ONLY the context provided. "
    "If the context does not contain enough information, say so clearly. "
    "Do not make up facts."
)

def make_question(chunk: dict) -> str:
    title = chunk["metadata"].get("title", "")
    date  = chunk["metadata"].get("date", "")[:10]
    text  = chunk["text"]
    mentioned = [p for p in PROVINCES if p.lower() in text.lower()]
    location  = mentioned[0] if mentioned else "the affected region"
    if title:
        return f"Based on the situation report '{title}' ({date}), what was the humanitarian situation in {location}?"
    return f"What does this humanitarian situation report say about {location}?"


def extract_answer(chunk: dict) -> str:
    sentences = re.split(r'(?<=[.!?])\s+', chunk["text"].strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
    factual = [s for s in sentences if re.search(r'\d', s)]
    selected = factual[:3] if factual else sentences[:3]
    return " ".join(selected)


def format_example(chunk: dict) -> str:
    """Wrap question + context + answer in Llama 3.1's chat template."""
    question = make_question(chunk)
    context  = chunk["text"]
    answer   = extract_answer(chunk)
    user_content = f"Context:\n{context}\n\nQuestion: {question}"

    return (
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{answer}<|eot_id|>"
    )


examples = [format_example(c) for c in chunks]

print(f"Formatted {len(examples)} training examples")
print("\n--- Example ---")
print(examples[0])

## Cell 5 — Build the HuggingFace Dataset

SFTTrainer expects a HuggingFace Dataset object with a text column.
We split off 5% for validation so we can monitor the loss on unseen examples
and catch overfitting early.

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({"text": examples})
dataset = dataset.train_test_split(test_size=0.05, seed=42)

print(f"Train examples: {len(dataset['train'])}")
print(f"Val   examples: {len(dataset['test'])}")

## Cell 6 — Load Llama 3.1 8B in 4-bit

**BitsAndBytesConfig** tells the library how to compress the weights:
- `load_in_4bit=True` — compress from 32-bit to 4-bit (8× smaller)
- `bnb_4bit_quant_type="nf4"` — NormalFloat4, the quantisation format that
  loses the least information for normally-distributed neural network weights
- `bnb_4bit_compute_dtype=float16` — even though weights are stored in 4-bit,
  the actual matrix multiplications run in float16 for numerical stability
- `bnb_4bit_use_double_quant=True` — quantise the quantisation constants too,
  saving another ~0.4 bits per parameter

Llama 3.1 8B in 4-bit uses ~5 GB of VRAM, well within the T4's 16 GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokeniser...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # pad on the right for causal LMs

print("Loading model in 4-bit (this may take a minute)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False  # disable KV cache during training

print(f"Model loaded — parameters: {model.num_parameters():,}")

## Cell 7 — Attach LoRA adapters

**LoraConfig** controls the adapter architecture:
- `r=16` — rank of the adapter matrices. Higher = more parameters = better
  quality but slower. 16 is a good balance for a T4.
- `lora_alpha=32` — scaling factor. Rule of thumb: set to 2× r.
- `target_modules` — which layers get adapters. For Llama 3.1 we target
  both the attention projections (q, k, v, o) and the MLP layers
  (gate, up, down), which gives better domain adaptation than attention alone.
- `lora_dropout=0.05` — randomly zero out adapter outputs during training
  to prevent overfitting.
- `task_type="CAUSAL_LM"` — tells PEFT this is a text generation task.

After `get_peft_model`, the base weights are completely frozen.
Only the adapter weights (~40M out of 8B, ~0.5%) will be updated by AdamW.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention
        "gate_proj", "up_proj", "down_proj",       # MLP
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: trainable params ~40M out of ~8B (~0.5%)

## Cell 8 — Train with SFTTrainer

**TrainingArguments** key hyperparameters:
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=8` —
  effective batch size of 16. We use a smaller per-device batch than Phi-3
  because Llama 3.1 8B is twice as large.
- `num_train_epochs=3` — pass through the full dataset 3 times.
- `learning_rate=2e-4` — standard for LoRA fine-tuning.
- `fp16=True` — adapter weights and gradients in float16, saves memory.
- `warmup_ratio=0.03` — gradually ramp up the learning rate for the first
  3% of steps to prevent instability at the start.
- `evaluation_strategy="epoch"` — measure validation loss after each epoch.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./geointel-llama3-adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none",
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=512,
    args=training_args,
)

print("Starting training...")
trainer.train()

## Cell 9 — Push adapter to HuggingFace Hub

Only the LoRA adapter weights (~80 MB) are pushed — not the full 8B model.
Anyone loading the adapter downloads the base Llama 3.1 8B from Meta/HF
separately and merges the adapter on top at inference time.

**Note:** The production app uses the base Llama 3.1 8B via Groq.
To serve this fine-tuned adapter in production you would need a dedicated
inference endpoint (e.g. HuggingFace Inference Endpoints, Together.ai).

In [ ]:
print(f"Pushing adapter to {ADAPTER_REPO} ...")
trainer.model.push_to_hub(ADAPTER_REPO)
tokenizer.push_to_hub(ADAPTER_REPO)
print("Done — adapter available at:", f"https://huggingface.co/{ADAPTER_REPO}")

## Cell 10 — Quick inference test

Test the fine-tuned adapter before leaving Colab.
This runs generation directly in the notebook — no API call needed.

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
)

test_prompt = (
    "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    "You are a humanitarian intelligence assistant specialising in "
    "disaster response for the 2023 Turkey-Syria earthquake. "
    "Answer using ONLY the context provided.<|eot_id|>"
    "<|start_header_id|>user<|end_header_id|>\n\n"
    "Context:\n"
    "As of 10 February 2023, Hatay province recorded 847 destroyed buildings. "
    "Over 400,000 people remain displaced across the 11 affected provinces. "
    "Search and rescue operations concluded after 14 days.\n\n"
    "Question: What is the humanitarian situation in Hatay?<|eot_id|>"
    "<|start_header_id|>assistant<|end_header_id|>\n\n"
)

output = pipe(test_prompt)[0]["generated_text"]
answer = output[len(test_prompt):].split("<|eot_id|>")[0].strip()
print("Answer:", answer)